# 01. Train experiments

이 노트북은 학습만 담당합니다. 아래 설정을 바꾼 뒤 실행하면 결과와 checkpoint가 experiment/seed별 파일로 저장됩니다. 그래프와 표는 `02_analyze_experiments.ipynb`에서 생성합니다.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (Path('D:/gt-super') / 'data').exists():
    ROOT = Path('D:/gt-super')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from gt_aux.config import ExperimentConfig
from gt_aux.data import prepare_data
from gt_aux.train import release_model, train_one_experiment

## 학습 설정

여러 seed 실험은 `SEED`만 바꾸어 노트북을 다시 실행합니다. 같은 seed 집합을 모든 experiment에 사용해야 공정하게 비교할 수 있습니다.

In [ ]:
RUN_MODE = 'smoke'
SEED = 43
EXPERIMENTS = ['baseline', 'shared_detach', 'shared_e2e']

TRAIN_IMAGES = 400
VAL_IMAGES = 100
EPOCHS = 7
BATCH_SIZE = 2
NUM_WORKERS = 0

IMAGE_MIN_SIZE = 384
IMAGE_MAX_SIZE = 640
LEARNING_RATE = 2e-4
BACKBONE_LEARNING_RATE = 2e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 0.1
AUX_WEIGHT = 0.5
FEATURE_LEVEL = 0
HORIZONTAL_FLIP_P = 0.5
USE_AMP = None  # CUDA에서는 자동 활성화, CPU에서는 자동 비활성화
SAVE_EPOCH_CHECKPOINTS = False  # True면 epoch별 checkpoint가 추가로 쌓임

CONFIG = ExperimentConfig.for_run(
    ROOT, run_mode=RUN_MODE, seed=SEED, experiments=EXPERIMENTS,
    train_images=TRAIN_IMAGES, val_images=VAL_IMAGES, epochs=EPOCHS,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    image_min_size=IMAGE_MIN_SIZE, image_max_size=IMAGE_MAX_SIZE,
    lr=LEARNING_RATE, backbone_lr=BACKBONE_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
    base_aux_weight=AUX_WEIGHT, feature_level=FEATURE_LEVEL,
    horizontal_flip_p=HORIZONTAL_FLIP_P, use_amp=USE_AMP,
    save_epoch_checkpoints=SAVE_EPOCH_CHECKPOINTS,
)
CONFIG.as_dict()

In [ ]:
BUNDLE = prepare_data(CONFIG)
print({'train_images': len(BUNDLE.train_records), 'val_images': len(BUNDLE.val_records)})

## 학습 실행

`EXPERIMENTS`에 있는 모델을 순서대로 처음부터 학습합니다. 기존 동일 experiment/seed 결과가 있으면 덮어쓰므로, 재실행 전에 `SEED`를 확인하세요.

In [ ]:
for experiment in CONFIG.experiments:
    model, history, gradients = train_one_experiment(
        CONFIG, BUNDLE, experiment=experiment, seed=CONFIG.seed
    )
    release_model(model)
    print({
        'experiment': experiment,
        'seed': CONFIG.seed,
        'checkpoint': str(CONFIG.checkpoint_path(experiment)),
        'final_mAP': float(history.iloc[-1]['map']),
    })